# Reasoning Techniques

From chain-of-thought prompting to native reasoning models with thinking budgets.

This notebook is an original tutorial (rewritten for depth, 2026 practice).

## Learning Objectives

- Explain CoT, self-consistency, and tree-of-thought mechanically — including when each pays.
- Describe ReAct and PAL/program-aided reasoning in one line each.
- Explain reasoning models (extended thinking) and how they change prompting: thinking budgets, when explicit CoT prompting is redundant.
- Choose a technique by problem shape: verifiable vs open-ended, single-path vs search.

## Mental Model

All reasoning techniques buy accuracy by **spending more inference compute
per answer** — test-time scaling. They differ in the *shape* of the spend:

| Technique | Shape of extra compute |
|---|---|
| CoT | one longer path (reason before answering) |
| Self-consistency | k independent paths + vote |
| Tree-of-thought | branching search with evaluation and backtracking |
| ReAct | interleave reasoning with tool observations |
| PAL | delegate the calculation to code |
| Reasoning models | the model spends its own hidden thinking tokens, budget-controlled |

The 2026 shift: reasoning that used to be prompted (CoT) is now *trained in*
and metered (thinking budgets). Your job moved from "elicit reasoning" to
"allocate reasoning" — deciding how much thinking each request deserves.

## Important Concepts

- **Chain-of-thought (CoT)**: ask the model to reason step-by-step before answering. Big gains on multi-step problems for non-reasoning models; on reasoning models it's largely redundant (they think natively) and can even interfere with their trained reasoning process.
- **Self-consistency**: sample k CoT paths at temperature > 0, take the majority answer. Turns variance into signal; only works when answers are comparable (math, classification — not essays). Cost: k x.
- **Tree-of-thought (ToT)**: generate multiple candidate *steps*, evaluate each ("sure/maybe/dead-end"), expand the promising ones, backtrack from dead ends. A search algorithm with the LLM as both generator and evaluator. Pays on puzzles/planning with dead ends; overkill for linear problems.
- **ReAct**: Thought → Action(tool) → Observation loop; reasoning grounded in external evidence (notebook 05).
- **PAL / program-aided**: model writes code for the calculation; the runtime executes it. Arithmetic/date/logic errors drop to ~zero — delegate what a computer does better.
- **Reasoning models / extended thinking**: models trained (RL on verifiable rewards — see GRPO, notebook 30) to produce long internal reasoning before answering. Controlled by a **thinking budget / reasoning effort** parameter. Tradeoff: latency and cost scale with thinking; simple queries don't deserve max effort.
- **Test-time scaling law**: accuracy improves with inference compute (longer thinking, more samples, search) — the complement to train-time scaling.

## Need-To-Know Coverage Checklist

- [x] CoT mechanics and when it's now redundant (reasoning models).
- [x] Self-consistency: sample-k + majority vote, in code.
- [x] ToT: propose/evaluate/backtrack search mechanics.
- [x] ReAct and PAL, one line each, with pointers.
- [x] Reasoning models, thinking budgets, effort allocation.
- [x] Choosing by problem shape; cost/latency of each technique.

## Deep Study Notes

### CoT and its 2026 status

"Let's think step by step" and few-shot CoT exemplars raised multi-step
accuracy dramatically on instruction-tuned models. Two caveats that read as
current knowledge: (1) CoT text is not a faithful trace of the model's
computation — it's a useful artifact, not an explanation; (2) on native
reasoning models, prompted CoT is mostly redundant — the model already
deliberates internally — and vendors advise *against* heavy CoT scaffolding
that fights the trained behavior. Say "for non-reasoning models, CoT; for
reasoning models, set the budget."

### Self-consistency

Sample k reasoning paths (temperature ~0.7), extract each final answer,
majority-vote. Why it works: individual paths make uncorrelated errors; the
mode of the answer distribution is more reliable than any single draw.
Practical notes: k=5-10 captures most of the gain; requires an extractable,
comparable answer; agreement rate doubles as a **confidence signal**
(notebook 26 uses exactly this for classification confidence).

### Tree-of-thought

ToT = beam search over partial solutions:
1. **Propose** b candidate next steps from each frontier state.
2. **Evaluate** each candidate (the LLM scores: sure / maybe / impossible).
3. **Prune** dead ends, keep the best w (beam width), recurse.
4. **Backtrack** when a branch exhausts.

Costs multiply fast (depth x branching x evaluation calls) — reserve for
problems with genuine dead ends (games, constrained planning, some code
search). For linear reasoning, CoT or a reasoning model wins on cost.

### Reasoning models and budget allocation

Trained via RL on verifiable rewards to "think" (hidden tokens) before
answering. The API lever is a thinking/effort budget. Production allocation
pattern: route by task difficulty — low effort for lookups and formatting,
high effort for math/planning/debugging — the same cascade logic as model
routing (notebook 30). Two gotchas: thinking tokens are billed and slow
(latency budget in voice loops — notebook 23 — mostly excludes them), and
long thinking does not fix missing information (retrieval beats rumination).

## Common Failure Modes

- Heavy CoT scaffolding on a reasoning model → fights trained behavior, wastes tokens.
- Self-consistency on open-ended output → no comparable answers to vote on.
- ToT on linear problems → 10x cost, no accuracy gain.
- Max thinking budget on every request → latency and bill blow up for lookups.
- Trusting CoT text as a faithful explanation → miscalibrated confidence in audits.
- Rumination in place of retrieval → thinking harder about facts the model doesn't have.

## Implementation Notes

- Route effort: classify request difficulty, map to thinking budget / technique.
- For math/dates/counts inside any pipeline, prefer PAL (generate code, execute) over verbal arithmetic.
- Log answers AND agreement rates when using self-consistency — the agreement trend is a drift signal.
- Verifiable tasks: pair any technique with a checker (tests, schema, calculator) — verification beats more reasoning.

In [ ]:
"""Self-consistency and a toy tree-of-thought search, both runnable.

The fake sampler simulates the answer distribution of a mid-tier model on
a multi-step problem: mostly right, sometimes wrong in varied ways.
"""
import random
from collections import Counter

# --- 1. self-consistency: sample k paths, majority vote -------------------
def sample_answer(rng):
    # 60% correct answer, 40% spread across distinct wrong answers
    return rng.choices(["42", "38", "56", "40"], weights=[60, 15, 15, 10])[0]


def self_consistency(k, seed=7):
    rng = random.Random(seed)
    answers = [sample_answer(rng) for _ in range(k)]
    votes = Counter(answers)
    best, n = votes.most_common(1)[0]
    return {"answer": best, "agreement": n / k, "votes": dict(votes)}


for k in (1, 5, 15):
    r = self_consistency(k)
    print(f"k={k:>2}: answer={r['answer']} agreement={r['agreement']:.0%} votes={r['votes']}")
# Single sample is wrong 40% of the time; the k=15 vote is reliably right,
# and 'agreement' doubles as a confidence signal.

# --- 2. tree-of-thought: propose / evaluate / prune / backtrack ------------
# Toy task: build the sequence [2, 4, 8, 16] by proposing next numbers.
TARGET = [2, 4, 8, 16]


def propose(state):                      # generator: candidate next steps
    last = state[-1] if state else 1
    return [state + [last * 2], state + [last + 2], state + [last * 3]]


def evaluate(state):                     # evaluator: sure / maybe / dead-end
    if state == TARGET[: len(state)]:
        return "sure"
    return "dead-end"


def tot_search(beam_width=2, max_depth=4):
    frontier, explored = [[2]], 0
    for depth in range(max_depth - 1):
        candidates = [s for st in frontier for s in propose(st)]
        explored += len(candidates)
        scored = [(s, evaluate(s)) for s in candidates]
        frontier = [s for s, v in scored if v == "sure"][:beam_width]  # prune
        if not frontier:
            return None, explored
    return frontier[0], explored


solution, explored = tot_search()
print(f"\nToT found {solution} after evaluating {explored} candidates "
      "(pruned dead ends instead of enumerating all 27 paths)")


## Practice

1. Lower the correct-answer weight to 40% and rerun self-consistency. At what
   k does the vote stop being reliable, and what does that say about using
   agreement as confidence?
2. Add a "maybe" verdict to `evaluate` (state matches except the last item)
   and make the search keep maybes only when no sures exist — that's the
   backtracking heuristic.
3. For each task, pick the cheapest adequate technique and defend it:
   (a) "what's 14 business days after March 3?", (b) triage 10k support
   tickets, (c) find the bug in this race condition, (d) solve this
   scheduling puzzle with hard constraints.
4. Your voice agent (notebook 23) needs occasional deep reasoning. Where does
   the thinking happen so the latency budget survives?

## Design Checklist

- [ ] Technique matched to problem shape; no ToT on linear tasks.
- [ ] Reasoning models: budget routed by difficulty, not maxed globally.
- [ ] No prompted-CoT scaffolding fighting a reasoning model.
- [ ] Self-consistency only where answers are extractable and comparable; agreement logged.
- [ ] Calculations delegated to code (PAL), not verbal arithmetic.
- [ ] Verification (tests/schema/checker) paired with reasoning wherever the task is verifiable.